# Hourly Transform — Bronze → Silver
Reads one hour's JSON from `bronze/hourly/YYYY-MM-DD/HH/air_quality.json`,
flattens it, cleans it, and appends Parquet to `silver/hourly/`.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BUCKET = 'hcm-air-quality-486008'
SA_KEY = os.path.abspath('../keys/hcm-pipeline-sa.json')

# Change these to match a file that actually exists in your bronze/hourly/
DATE = '2026-04-08'
HOUR = '5'

print('SA key path:', SA_KEY)
print('Key exists:', os.path.exists(SA_KEY))
print('Processing:', DATE, 'hour', HOUR)

SA key path: d:\data-engineering-zoomcamp\project\keys\hcm-pipeline-sa.json
Key exists: True
Processing: 2026-04-08 hour 5


In [2]:
spark = SparkSession.builder \
    .appName('hcm-hourly') \
    .config('spark.jars.packages', 'com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.22') \
    .config('spark.hadoop.google.cloud.auth.service.account.enable', 'true') \
    .config('spark.hadoop.google.cloud.auth.service.account.json.keyfile', SA_KEY) \
    .config('spark.hadoop.fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem') \
    .config('spark.hadoop.fs.AbstractFileSystem.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Spark version: 4.1.1


In [3]:
spark

In [4]:
# Read the raw JSON
# multiline=true because the whole JSON object spans multiple lines

src = f"gs://{BUCKET}/bronze/hourly/{DATE}/{HOUR.zfill(2)}/air_quality.json"
print("Reading:", src)

df = spark.read.option("multiline", "true").json(src)

df.printSchema()

Reading: gs://hcm-air-quality-486008/bronze/hourly/2026-04-08/05/air_quality.json
root
 |-- carbon_monoxide: double (nullable = true)
 |-- european_aqi: long (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- nitrogen_dioxide: double (nullable = true)
 |-- observed_at: string (nullable = true)
 |-- ozone: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)
 |-- sulphur_dioxide: double (nullable = true)
 |-- us_aqi: long (nullable = true)



In [ ]:
# See what the raw data looks like 
# Flat JSON — one record per file, direct fields
df.limit(5).toPandas()

,carbon_monoxide,european_aqi,ingested_at,latitude,longitude,nitrogen_dioxide,observed_at,ozone,pm10,pm2_5,sulphur_dioxide,us_aqi
0,467.0,64,2026-04-08T05:00:08.006686+00:00,10.800003,106.600006,12.1,2026-04-08T12:00,142.0,30.4,28.9,8.9,87


In [6]:
# Keep only the columns we need, rename observed_at → timestamp
df = df.select(
    F.col("observed_at").alias("timestamp"),
    F.col("pm10"),
    F.col("pm2_5"),
    F.col("carbon_monoxide"),
    F.col("nitrogen_dioxide"),
    F.col("sulphur_dioxide"),
    F.col("ozone"),
    F.col("us_aqi").cast("double"),
    F.col("european_aqi").cast("double"),
)



In [9]:
df = df.withColumn("date", F.to_date(F.col("timestamp")))
df = df.withColumn("hour", F.hour(F.to_timestamp(F.col("timestamp"))))
df = df.withColumn("ingested_at", F.current_timestamp())

df.show()

+----------------+----+-----+---------------+----------------+---------------+-----+------+------------+----------+----+--------------------+
|       timestamp|pm10|pm2_5|carbon_monoxide|nitrogen_dioxide|sulphur_dioxide|ozone|us_aqi|european_aqi|      date|hour|         ingested_at|
+----------------+----+-----+---------------+----------------+---------------+-----+------+------------+----------+----+--------------------+
|2026-04-08T12:00|30.4| 28.9|          467.0|            12.1|            8.9|142.0|  87.0|        64.0|2026-04-08|  12|2026-04-10 15:50:...|
+----------------+----+-----+---------------+----------------+---------------+-----+------+------------+----------+----+--------------------+



In [10]:
# Write to silver — append mode so each hour adds one row
# partitionBy('date') creates subfolders like date=2026-04-08/
# which makes BigQuery loading faster later

dest = f'gs://{BUCKET}/silver/hourly/'

df.write.mode('append').partitionBy('date').parquet(dest)

print(f'✅ Done — date={DATE}, hour={HOUR.zfill(2)}')



✅ Done — date=2026-04-08, hour=05
